In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Lab 9 - Multi-layer Perceptron Forward Pass & Backpropagation

## Part I
For this exercise you will implement a simple 2-layer perceptron with the forward pass and the backpropagation to learn the weights

For the first part you'll build and train a 2-layer neural network that predicts the prices of houses, using the usual Boston housing dataset.

In [3]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
boston = pd.read_table("housing.txt", names=housing_names, sep=r'\s+')

As usual, consider the MEDV as your target variable. 
* Split the data into training, validation and testing (70,15,15)%
* Experiment with different number of neurons per layer for your network, using the validation set

In [10]:
# your code goes here
y = boston.values[:,-1]
X = boston.values[:,:-1]

X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.3, random_state=12)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5, random_state=12)

In [5]:
def sigmoid_activation(z):
    # your code goes here
    return 1/(1 + np.exp(-z))

def add_bias(X:np.ndarray):
    return np.c_[X, np.ones((X.shape[0], 1))]

In [87]:
# your code goes here
def initialize_two_layer_perceptron(X: np.ndarray, act_hidden, dim_input, dim_hidden, dim_output, act_output=None):
    """
    Implements the forward pass of a two-layer fully connected perceptron.
    
    Parameters
    ----------
    X : a 2-dimensional array
        the input data
    act_hidden : function
        the activation function to be used for the hidden layer
    act_output : function
        the activation function to be used for the output layet
    dim_input : int
        the dimensionality of the input layer
    dim_hidden : int
        the dimensionality of the hidden layer
    dim_output : int
        the dimensionality of the output layer
    Returns
    -------
    y_pred : float
        the output of the computation of the forward pass of the network
    """

    # hidden layer
    X_bb = add_bias(X) # bb = by bias
    W1 = np.random.randn(dim_input + 1, dim_hidden) # plus 1 for bias
    A1 = act_hidden(X_bb.dot(W1))

    # saida
    A1_bb = add_bias(A1)
    W2 = np.random.randn(dim_hidden + 1, dim_output)
    Z2 = A1_bb.dot(W2) 

    if act_output != None:
        A2 = act_output(Z2)
        return W1, A1, W2, A2 # W, A and Z matrix

    return  W1, A1_bb, W2, Z2 # W, A and Z matrix

def forward_pass_two_layers(X: np.ndarray, W1, act_hidden, W2, act_output=None):
    """
    Parameters
    ----------

    X: a 2-dimensional array
        the input data without bias
    W1, W2: a 2-dimensional array
        the weights for layer 1 and layer 2, by bias
    """
    X_bb = add_bias(X)
    A1 = act_hidden(X_bb.dot(W1))

    A1_bb = add_bias(A1)
    Z2 = A1_bb.dot(W2)

    if act_output != None:
        A2 = act_hidden(Z2)
        return A2

    return Z2
    
    

In [88]:
W1, A1, W2, Y_pred = initialize_two_layer_perceptron(
    X_train,
    sigmoid_activation,
    X_train.shape[1],
    20,
    1
)

C:\Users\C3007803\AppData\Local\Temp\ipykernel_10488\1478924477.py:3: RuntimeWarning: overflow encountered in exp
  return 1/(1 + np.exp(-z))


In [89]:
A1.shape[1]

21

In [97]:
# Y_pred = A2

def backpropagation_GD(X, W1, A1, W2, Y_pred, Y_true):

    # Delta de todos os neuronios da output layer
    dla_out = 2*(Y_true - Y_pred) * Y_pred * (1 - Y_pred)

    dla_hidden = (W2).dot(dla_out) * A1 * (1 - A1)
    #for k in range(Y_pred.shape[1]): #dim of output
    #    dla_hidden += dla_out * W2[0, k] * A1[0,:] * (1 - A1[0,:])

    #dla_input = 0
    #for k in range(A1.shape[1]): #dim of hidden
    #    dla_input += dla_hidden[k] * W1[0, k] * X[0, :] * (1 - X[0,:])

    return dla_out, dla_hidden


In [98]:
pepe, pepa =backpropagation_GD(X_train, W1, A1, W2, Y_pred, y_train[:,None])

ValueError: shapes (21,1) and (354,1) not aligned: 1 (dim 1) != 354 (dim 0)

In [92]:
pepe, pepa

(array([-474.57032718]),
 array([[-9.70734982e-096,  1.08137410e-014, -0.00000000e+000, ...,
         -0.00000000e+000, -0.00000000e+000, -0.00000000e+000],
        [-2.63659393e-133,  3.65312208e-128, -0.00000000e+000, ...,
         -0.00000000e+000, -1.34762014e-107, -0.00000000e+000],
        [-6.74031586e-134,  6.70480892e-061, -0.00000000e+000, ...,
         -0.00000000e+000, -0.00000000e+000, -0.00000000e+000],
        ...,
        [-9.16732649e-167,  2.45224855e-131, -0.00000000e+000, ...,
         -0.00000000e+000, -8.51109478e-055, -0.00000000e+000],
        [-2.73758532e-076,  7.83363356e-118, -3.58705445e-016, ...,
         -0.00000000e+000, -1.74746474e-118, -0.00000000e+000],
        [-5.04688660e-100,  1.88633350e-011, -0.00000000e+000, ...,
         -0.00000000e+000, -0.00000000e+000, -0.00000000e+000]],
       shape=(354, 21)))

## Part II 

For this exercise you will build and train a 2-layer neural network that predicts the exact digit from a hand-written image, using the MNIST dataset. 
For this exercise, add weight decay to your network.

In [14]:
from sklearn.datasets import load_digits

In [15]:
digits = load_digits()

In [17]:
X = digits.data
y = digits.target

In [19]:
X.shape

(1797, 64)

Again, you will split the data into training, validation and testing.

In [ ]:
# your code goes here:


In [ ]:
# your code goes here:
